# Audited revision (v5)

Original files are preserved. All saved outputs in this revised notebook are cleared so old results cannot be mistaken for rerun results. The original full notebook and its recorded outputs remain the provenance source.

Fixes include explicit saved-model keys, consistency checks against stored errors, exact calibration bounds, correct residual-score documentation, fail-closed boundary sample selection, and corrected statistical interpretations. The full notebook retains the explicit seed 2026 for the original development split.

The accompanying reproduction package distinguishes reconstructed model artifacts from the unavailable original time-stamped model file. New resolution and mechanism diagnostics are exploratory. Run the original-data cells in Colab with the same source files; never substitute a new model for the original frozen evaluation.


# Section 22: paper fixes (frozen figure, coverage check, E5 refit)

Runs on its own. It only **reads** earlier outputs and writes to `HIT_outputs_verified/paper_fixes/<timestamp>/`. Nothing in the frozen run is refitted or overwritten.

1. **Fig. 1 re-export**: `frozen_errors.png` (300 dpi) + `frozen_errors.pdf` (vector), one IEEE column wide.
2. **Coverage check**: tests whether equal coverage for uncorrected and global correction (96.4/96.4, 90.0/90.0) is real. The global correction divides by one constant, so its residual is (e_raw − c)/(1 + c/100). With almost all raw errors positive, both methods end up covering the same raw-error range. The script measures this and lists any object where the two disagree.
3. **E5 refit**: the same 60 objects, with (a) exact point-to-triangle distances alongside the original 32-centroid approximation and (b) a maximum-likelihood logistic fit, accepted only if it converges, is not on a bound, and agrees across 5 starts.

Runtime: steps 1–2 under a minute. Step 3 needs the extracted development shard (unzip about 6 min if not already on local disk), then about 10–40 s per object. It checkpoints after every object, so rerun the cell if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob
from datetime import datetime, timezone

OUT_ROOT  = '/content/drive/MyDrive/HIT_outputs_verified'
DEV_ZIP   = '/content/drive/MyDrive/10_24553_27272.zip'   # development shard archive
DATA_DIR  = '/content/data'                                 # local disk (fast)

# Frozen run used in the paper; falls back to the newest one if this folder is missing.
FROZEN_DIR = os.path.join(OUT_ROOT, 'frozen_validation_1_1_2729', '20260924T193719_753199Z')
if not os.path.isdir(FROZEN_DIR):
    raise FileNotFoundError("Set FROZEN_DIR to the exact original frozen run; no newest-run fallback")

# Earlier Section 20 run (to reuse the same 60 E5 object IDs).
PREV_DIR = os.path.join(OUT_ROOT, 'remaining_analyses', '20260924T201209Z')
if not os.path.isdir(PREV_DIR):
    raise FileNotFoundError("Set PREV_DIR to the exact original boundary run; no newest-run fallback")

OUT = os.path.join(OUT_ROOT, 'paper_fixes', datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
os.makedirs(OUT, exist_ok=True)
print('frozen run :', FROZEN_DIR)
print('previous E5:', PREV_DIR)
print('output     :', OUT)

In [ ]:
!pip -q install trimesh

In [ ]:
%%writefile /content/frozen_figure_and_coverage.py
#!/usr/bin/env python3
"""Paper fixes 1 and 2 (frozen second-shard outputs).

1. Re-export the frozen-error box plot (Fig. 1) at 300 dpi PNG plus vector PDF,
   sized for one IEEE column (3.5 in).
2. Check the identical coverage values in Table IV (uncorrected vs. global
   correction: 96.4 / 96.4 internal, 90.0 / 90.0 frozen).

Why identical coverage can be expected. The global correction divides every
occupancy volume by the same factor k = 1 + c/100, where c is the mean training
error. Its percentage residual is therefore (e_raw - c) / k, and an object is
covered when
        c - b_g*k  <=  e_raw  <=  c + b_g*k .
The uncorrected method covers an object when -b_u <= e_raw <= b_u. Because
almost every raw error is positive, and the calibration quantiles of |e_raw|
and |e_raw - c| usually come from the same calibration object, the upper limits
c + b_g*k and b_u nearly coincide. Both methods then cover almost exactly the
same objects. The script measures how closely this holds and lists every object
on which the two methods disagree.

Nothing here refits or changes the frozen model. It only reads saved outputs.
"""
import argparse, glob, json, os, re, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd

PAPER = {"bounds": {"uncorrected": 8.451, "global": 4.632, "occ_only": 2.661},
         "frozen_coverage": {"uncorrected": 90.0, "global": 90.0, "occ_only": 92.1},
         "internal_coverage": {"uncorrected": 96.4, "global": 96.4, "occ_only": 96.2}}
LABELS = {"uncorrected": "Uncorrected", "global": "Global", "occ_only": "Occupancy-only"}

# Column-name patterns for the signed percentage error of each method. The
# first matching column is used. Override with --col-uncorrected etc.
PATTERNS = {
    "uncorrected": [r"^(e|err|error|pct_err|signed_err)_?(raw|uncorr)", r"uncorr.*(err|pct|resid)",
                    r"(err|pct|resid).*uncorr", r"^raw_.*(err|pct)", r"(err|pct).*_raw$"],
    "global": [r"glob.*(err|pct|resid)", r"(err|pct|resid).*glob"],
    "occ_only": [r"occ.*only.*(err|pct|resid)", r"(err|pct|resid).*occ.*only", r"ols.*(err|pct|resid)",
                 r"(err|pct|resid).*ols", r"occupancy_only"],
}
EXCLUDE = re.compile(r"abs|ape|within|cover|pass|bound|_ci|lo$|hi$|pred|hat", re.I)
ID_PAT = re.compile(r"^(instance_?id|object_?id|id|instance)$", re.I)


def find_tables(root, exclude=None):
    out = []
    for p in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True) + \
             glob.glob(os.path.join(root, "**", "*.parquet"), recursive=True):
        if exclude and re.search(exclude, p):
            continue
        try:
            df = pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p)
        except Exception:
            continue
        out.append((p, df))
    return out


def detect_cols(df, overrides):
    cols = {}
    for m, pats in PATTERNS.items():
        if overrides.get(m):
            cols[m] = overrides[m]
            continue
        hit = None
        for pat in pats:
            cands = [c for c in df.columns if re.search(pat, str(c), re.I) and not EXCLUDE.search(str(c))
                     and pd.api.types.is_numeric_dtype(df[c])]
            if cands:
                hit = cands[0]
                break
        cols[m] = hit
    return cols


def pick_table(root, overrides, min_rows, explicit=None, exclude=None):
    tables = [(explicit, pd.read_csv(explicit))] if explicit else find_tables(root, exclude)
    best = None
    for p, df in tables:
        if len(df) < min_rows:
            continue
        cols = detect_cols(df, overrides)
        score = sum(v is not None for v in cols.values())
        if best is None or score > best[0] or (score == best[0] and len(df) > len(best[2])):
            best = (score, p, df, cols)
    return best


def flatten(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from flatten(v, f"{prefix}/{k}")
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            yield from flatten(v, f"{prefix}[{i}]")
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        yield prefix, float(obj)


def find_bounds_and_offset(roots):
    """Read named fields from one model, never search summary statistics by regex."""
    mapping = {"uncorrected":"uncorrected", "global":"global_train_mean", "occ_only":"occupancy_only_ols"}
    for root in roots:
        for path in sorted(glob.glob(os.path.join(root, "**", "validation_v3_model.json"), recursive=True)):
            with open(path) as stream:
                model = json.load(stream)
            if "calibration_bounds_pct" not in model or "global_train_mean_bias_pct" not in model:
                continue
            bounds = {k:(float(model["calibration_bounds_pct"][key]), f"{path}:calibration_bounds_pct/{key}")
                      for k,key in mapping.items()}
            offset = float(model["global_train_mean_bias_pct"])
            if not np.isfinite(offset) or offset <= -100:
                raise ValueError("Invalid saved global correction")
            if any(not np.isfinite(v[0]) or v[0] < 0 for v in bounds.values()):
                raise ValueError("Invalid saved calibration bounds")
            return bounds, (offset, f"{path}:global_train_mean_bias_pct")
    return {}, None


def coverage_check(df, cols, bounds, c_json, tag):
    e_u, e_g, e_o = (df[cols[m]].to_numpy(float) for m in ("uncorrected", "global", "occ_only"))
    b_u, b_g, b_o = (bounds[m] for m in ("uncorrected", "global", "occ_only"))
    cov_u, cov_g, cov_o = np.abs(e_u) <= b_u, np.abs(e_g) <= b_g, np.abs(e_o) <= b_o
    n = len(df)
    # Offset implied by the stored errors: e_g = (e_u - c)/(1 + c/100)  =>  c = (e_u - e_g)/(1 + e_g/100)
    c_obj = (e_u - e_g) / (1 + e_g / 100)
    c_est = float(np.median(c_obj))
    algebra_spread = float(np.max(np.abs(c_obj - c_est)))
    if not np.isfinite(np.concatenate([e_u,e_g,e_o])).all():
        raise ValueError("Nonfinite signed errors are not allowed")
    if algebra_spread > 1e-6:
        raise ValueError("Stored global errors do not obey a single constant correction")
    if c_json is not None and abs(c_json-c_est) > 1e-6:
        raise ValueError("Saved global correction disagrees with the stored residuals")
    c = c_json if c_json is not None else c_est
    k = 1 + c / 100
    lo_g, hi_g = c - b_g * k, c + b_g * k
    both = int(np.sum(cov_u & cov_g)); only_u = int(np.sum(cov_u & ~cov_g)); only_g = int(np.sum(~cov_u & cov_g))
    ids = df[[x for x in df.columns if ID_PAT.match(str(x))][0]].astype(str).to_numpy() \
        if any(ID_PAT.match(str(x)) for x in df.columns) else np.arange(n).astype(str)
    disagree = pd.DataFrame({"id": ids, "e_uncorrected": e_u, "e_global": e_g,
                             "covered_uncorrected": cov_u, "covered_global": cov_g})[cov_u != cov_g]
    res = {
        "evaluation": tag, "n": n,
        "coverage_pct": {"uncorrected": 100 * cov_u.mean(), "global": 100 * cov_g.mean(), "occ_only": 100 * cov_o.mean()},
        "covered_counts": {"uncorrected": int(cov_u.sum()), "global": int(cov_g.sum()), "occ_only": int(cov_o.sum())},
        "cross_tab": {"covered_by_both": both, "uncorrected_only": only_u, "global_only": only_g,
                      "neither": int(np.sum(~cov_u & ~cov_g))},
        "global_offset_c": c, "global_offset_source": "saved JSON" if c_json is not None else "implied by stored errors",
        "global_offset_implied_by_errors": c_est,
        "max_deviation_from_global_formula": algebra_spread,
        "raw_error_window_uncorrected": [-b_u, b_u],
        "raw_error_window_global": [lo_g, hi_g],
        "upper_limit_difference_pp": hi_g - b_u,
        "n_raw_errors_below_global_lower_limit": int(np.sum(e_u < lo_g)),
        "n_raw_errors_in_gap_between_upper_limits": int(np.sum((e_u > min(b_u, hi_g)) & (e_u <= max(b_u, hi_g)))),
        "n_negative_raw_errors": int(np.sum(e_u < 0)),
    }
    same = only_u == 0 and only_g == 0
    res["verdict"] = (
        ("IDENTICAL covered sets. " if same else f"Covered sets differ on {only_u + only_g} object(s), "
         f"which offset in the coverage count ({only_u} vs {only_g}). ")
        + f"The global method covers raw errors in [{lo_g:.3f}, {hi_g:.3f}] and the uncorrected method covers "
          f"[{-b_u:.3f}, {b_u:.3f}]. The upper limits differ by {hi_g - b_u:+.3f} percentage points and "
          f"{res['n_raw_errors_below_global_lower_limit']} raw error(s) fall below the global lower limit, so the "
          "equal coverage is an arithmetic consequence of a one-signed error distribution, not a copy error."
        if algebra_spread < 1e-3 else
        "WARNING: the stored global errors do not follow e_g = (e_u - c)/(1 + c/100) for a single c "
        f"(max deviation {algebra_spread:.4f}). Check which columns were read before interpreting coverage.")
    return res, disagree


def make_figure(df, cols, out_dir, n_label):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.spines.top": False,
                         "axes.spines.right": False, "pdf.fonttype": 42, "ps.fonttype": 42})
    data = [df[cols[m]].dropna().to_numpy(float) for m in ("uncorrected", "global", "occ_only")]
    fig, ax = plt.subplots(figsize=(3.5, 2.4), layout="constrained")
    bp = ax.boxplot(data, showfliers=True, widths=0.55, patch_artist=True,
                    medianprops=dict(color="#bd632d", lw=1.4), whiskerprops=dict(lw=0.9),
                    capprops=dict(lw=0.9), boxprops=dict(lw=0.9))
    for b in bp["boxes"]:
        b.set_facecolor("#dce7ef"); b.set_edgecolor("#28648a")
    ax.axhline(0, color="#777777", ls="--", lw=0.8, zorder=0)
    ax.set_xticks([1, 2, 3], [LABELS[m] for m in ("uncorrected", "global", "occ_only")])
    ax.set_ylabel("Volume error relative to mesh (%)")
    ax.grid(axis="y", alpha=0.15); ax.set_axisbelow(True)
    stem = os.path.join(out_dir, "frozen_errors")
    fig.savefig(stem + ".pdf", bbox_inches="tight")
    fig.savefig(stem + ".png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    stats = {}
    for m, x in zip(("uncorrected", "global", "occ_only"), data):
        q1, med, q3 = np.percentile(x, [25, 50, 75]); iqr = q3 - q1
        stats[m] = {"n": int(len(x)), "q1": q1, "median": med, "q3": q3,
                    "whisker_lo": float(x[x >= q1 - 1.5 * iqr].min()), "whisker_hi": float(x[x <= q3 + 1.5 * iqr].max()),
                    "n_beyond_whiskers": int(np.sum((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)))}
    return stem, stats


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--frozen-dir", required=True, help="frozen_validation_1_1_2729/<timestamp> folder")
    ap.add_argument("--frozen-table", help="per-object CSV (auto-detected if omitted)")
    ap.add_argument("--dev-dir", help="development outputs folder, for the internal-test check (optional)")
    ap.add_argument("--dev-table", help="development per-object CSV with a split column (optional)")
    ap.add_argument("--split-col", default=None, help="column holding train/cal/test (auto-detected)")
    ap.add_argument("--out", required=True)
    for m in PATTERNS:
        ap.add_argument(f"--col-{m.replace('_', '-')}", dest=f"col_{m}")
    for m in PATTERNS:
        ap.add_argument(f"--bound-{m.replace('_', '-')}", dest=f"bound_{m}", type=float)
    ap.add_argument("--global-offset", type=float, help="global correction c (%%), if not found in JSON")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    overrides = {m: getattr(a, f"col_{m}") for m in PATTERNS}

    best = pick_table(a.frozen_dir, overrides, 2000, a.frozen_table)
    if best is None or best[0] < 3:
        print("Could not find a per-object table with signed errors for all three methods.")
        if best:
            print("Best candidate:", best[1]); print("Columns:", list(best[2].columns))
        print("Rerun with --frozen-table and --col-uncorrected/--col-global/--col-occ-only.")
        sys.exit(1)
    _, tpath, fdf, cols = best
    print(f"Frozen table: {tpath}  (n={len(fdf)})")
    for m, c in cols.items():
        print(f"  {LABELS[m]:15s} <- column '{c}'  (mean {fdf[c].mean():+.3f}, median |e| {fdf[c].abs().median():.3f})")

    found, off = find_bounds_and_offset([a.frozen_dir] + ([a.dev_dir] if a.dev_dir else []))
    bounds, bsrc = {}, {}
    for m in PATTERNS:
        if getattr(a, f"bound_{m}") is not None:
            bounds[m], bsrc[m] = getattr(a, f"bound_{m}"), "command line"
        elif m in found:
            bounds[m], bsrc[m] = found[m]
        else:
            raise ValueError(f"Exact saved bound missing for {m}; supply the original model or an explicit bound")
        print(f"  bound {LABELS[m]:15s} = {bounds[m]:.6g}   [{bsrc[m]}]")
    c_json = a.global_offset if a.global_offset is not None else (off[0] if off else None)
    if off and a.global_offset is None:
        print(f"  global offset c = {off[0]:.6g}   [{off[1]}]")

    report = {"run_utc": datetime.now(timezone.utc).isoformat(), "frozen_table": tpath, "columns": cols,
              "bounds": bounds, "bound_sources": bsrc, "paper_values": PAPER}
    res, dis = coverage_check(fdf, cols, bounds, c_json, "frozen second shard")
    report["frozen"] = res
    dis.to_csv(os.path.join(a.out, "coverage_disagreements_frozen.csv"), index=False)

    # Optional internal-test check
    dev = None
    if a.dev_table:
        dev = pd.read_csv(a.dev_table)
    elif a.dev_dir:
        cand = pick_table(a.dev_dir, overrides, 400, exclude=r"frozen_validation|remaining_analyses|paper_fixes")
        if cand and cand[0] == 3:
            dev = cand[2]; print(f"Development table: {cand[1]}")
    if dev is not None:
        scol = a.split_col or next((c for c in dev.columns if re.search(r"split|partition|fold|set$", str(c), re.I)), None)
        if scol is None and 400 <= len(dev) <= 700:
            print(f"Development table has {len(dev)} rows and no split column; treating it as the internal test set.")
            test = dev
        elif scol is None:
            print("Development table has no split column; skipping internal-test check (use --split-col).")
            test = dev.iloc[0:0]
        else:
            test = dev[dev[scol].astype(str).str.lower().str.startswith("test")]
        if len(test):
            dcols = detect_cols(dev, overrides)
            if all(dcols.values()):
                r2, d2 = coverage_check(test.reset_index(drop=True), dcols, bounds, c_json, "internal test")
                report["internal_test"] = r2
                d2.to_csv(os.path.join(a.out, "coverage_disagreements_internal.csv"), index=False)

    stem, stats = make_figure(fdf, cols, a.out, len(fdf))
    report["figure"] = {"png": stem + ".png", "pdf": stem + ".pdf", "dpi": 300, "width_in": 3.5, "box_stats": stats}
    json.dump(report, open(os.path.join(a.out, "coverage_check.json"), "w"), indent=2, default=float)

    print("\n=== Coverage check ===")
    for key in ("internal_test", "frozen"):
        if key not in report:
            continue
        r = report[key]
        pv = PAPER["internal_coverage" if key == "internal_test" else "frozen_coverage"]
        print(f"\n{r['evaluation']} (n={r['n']})")
        for m in PATTERNS:
            print(f"  {LABELS[m]:15s} {r['coverage_pct'][m]:6.2f}%  ({r['covered_counts'][m]}/{r['n']})   paper: {pv[m]}%")
        print("  cross-tab:", r["cross_tab"])
        print("  " + r["verdict"])
    print(f"\nFigure written: {stem}.png (300 dpi) and {stem}.pdf")
    print("Copy frozen_errors.pdf into the Overleaf figures/ folder (main.tex can point to the .pdf).")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e5_boundary_refit.py
#!/usr/bin/env python3
"""Paper fix 3: rerun the E5 label-boundary fits with a convergence check.

For each object:
  * near-boundary occupancy query points are selected as in the original run
    (approximate distance from the 32 nearest triangle centroids, |d| < band,
    at most 3,000 points, fixed seed);
  * signed distances are computed twice: with the original 32-centroid
    approximation and EXACTLY (brute-force point-to-triangle distance over all
    faces, sign from the generalized winding number). This also answers the
    second E5 limitation, that the approximation was never validated;
  * P(inside | d) = 1 / (1 + exp((d - t50)/w)) is fitted by maximum likelihood
    (Bernoulli), from several starting points, and each fit is checked:
      - optimizer reports success
      - projected gradient small
      - t50 and w not on a parameter bound
      - all starts that reach the best likelihood agree on t50
      - both labels present (>= 20 points each) and the classes overlap
    Fits that fail any check are reported, not silently used.

Positive distance = outside the raw mesh, in normalized coordinates.
The volume-implied offset is t_vol = (V_o - V_m) / (A_m * s).
"""
import argparse, glob, json, os, re, sys, time
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from scipy.spatial import cKDTree
from scipy.stats import pearsonr, spearmanr

ORIGINAL = {"n": 60, "t50_median": 0.00304, "t50_iqr": [0.00285, 0.00317], "t50_cv": 0.09,
            "w_median": 0.00027, "t_vol_median": 0.00311, "ratio": 0.98, "per_object_r": 0.567}
PIPELINE = {"full_ray_offset": 1.5 / 256 / 0.9, "half_ray_offset": 0.5 * 1.5 / 256 / 0.9}


# ----------------------------------------------------------------- loading
def load_instance(d):
    import trimesh
    mesh = trimesh.load(os.path.join(d, "raw_mesh.off"), process=False, force="mesh")
    z = np.load(os.path.join(d, "occupancies.npz"))
    pts = z["points"].astype(np.float64)
    occ = z["occupancies"]
    if occ.dtype == np.uint8 and occ.size * 8 >= len(pts) and occ.size != len(pts):
        occ = np.unpackbits(occ)[: len(pts)]
    occ = occ.astype(bool).reshape(-1)
    loc = np.asarray(z["loc"], dtype=np.float64).reshape(3)
    scale = float(np.asarray(z["scale"]).reshape(()))
    return mesh, pts, occ, loc, scale


# ----------------------------------------------------------------- geometry
def closest_point_on_triangles(p, a, b, c):
    """Exact closest points (Ericson, Real-Time Collision Detection 5.1.5), vectorized.
    p: (m,1,3); a,b,c: (1,f,3) -> squared distances (m,f)."""
    ab, ac, ap = b - a, c - a, p - a
    d1 = np.einsum("...k,...k", ab, ap); d2 = np.einsum("...k,...k", ac, ap)
    bp = p - b
    d3 = np.einsum("...k,...k", ab, bp); d4 = np.einsum("...k,...k", ac, bp)
    cp = p - c
    d5 = np.einsum("...k,...k", ab, cp); d6 = np.einsum("...k,...k", ac, cp)
    va = d3 * d6 - d5 * d4; vb = d5 * d2 - d1 * d6; vc = d1 * d4 - d3 * d2
    denom = va + vb + vc
    with np.errstate(divide="ignore", invalid="ignore"):
        v = vb / denom; w = vc / denom
    q = a + ab * v[..., None] + ac * w[..., None]              # interior
    # edges
    with np.errstate(divide="ignore", invalid="ignore"):
        t_ab = d1 / (d1 - d3); t_ac = d2 / (d2 - d6); t_bc = (d4 - d3) / ((d4 - d3) + (d5 - d6))
    m_ab = (vc <= 0) & (d1 >= 0) & (d3 <= 0)
    m_ac = (vb <= 0) & (d2 >= 0) & (d6 <= 0)
    m_bc = (va <= 0) & ((d4 - d3) >= 0) & ((d5 - d6) >= 0)
    q = np.where(m_bc[..., None], b + (c - b) * t_bc[..., None], q)
    q = np.where(m_ac[..., None], a + ac * t_ac[..., None], q)
    q = np.where(m_ab[..., None], a + ab * t_ab[..., None], q)
    # vertices (checked last so they take precedence)
    q = np.where(((d6 >= 0) & (d5 <= d6))[..., None], np.broadcast_to(c, q.shape), q)
    q = np.where(((d3 >= 0) & (d4 <= d3))[..., None], np.broadcast_to(b, q.shape), q)
    q = np.where(((d1 <= 0) & (d2 <= 0))[..., None], np.broadcast_to(a, q.shape), q)
    diff = p - q
    return np.einsum("...k,...k", diff, diff)


def winding_number(P, V, F, chunk=None):
    """Generalized winding number (Jacobson et al. 2013) via the Van Oosterom-Strackee solid angle."""
    A, B, C = V[F[:, 0]][None], V[F[:, 1]][None], V[F[:, 2]][None]
    chunk = chunk or max(1, int(1.5e6 // len(F)))          # keeps memory near 0.5 GB
    out = np.empty(len(P))
    for i in range(0, len(P), chunk):
        p = P[i:i + chunk, None, :]
        a, b, c = A - p, B - p, C - p
        la, lb, lc = (np.linalg.norm(x, axis=-1) for x in (a, b, c))
        det = np.einsum("...k,...k", a, np.cross(b, c))
        den = la * lb * lc + np.einsum("...k,...k", a, b) * lc + np.einsum("...k,...k", b, c) * la \
            + np.einsum("...k,...k", c, a) * lb
        out[i:i + chunk] = np.arctan2(det, den).sum(axis=1) / (2 * np.pi)
    return out


def exact_unsigned(P, V, F, chunk=None):
    A, B, C = V[F[:, 0]][None], V[F[:, 1]][None], V[F[:, 2]][None]
    chunk = chunk or max(1, int(4e5 // len(F)))
    out = np.empty(len(P))
    for i in range(0, len(P), chunk):
        out[i:i + chunk] = np.sqrt(closest_point_on_triangles(P[i:i + chunk, None, :], A, B, C).min(axis=1))
    return out


def exact_unsigned_pruned(P, V, F, tree, upper):
    """Exact distance using only triangles that can possibly be closer than a known upper bound.
    A triangle within distance u of p has its centroid within u + r_max of p, where r_max is the
    largest centroid-to-vertex distance, so no candidate is missed."""
    tri = V[F]
    cent = tri.mean(axis=1)
    r_max = float(np.linalg.norm(tri - cent[:, None, :], axis=-1).max())
    out = np.empty(len(P))
    for i, (p, u) in enumerate(zip(P, upper)):
        idx = np.asarray(tree.query_ball_point(p, u + r_max + 1e-12), dtype=int)
        t = tri[idx]
        out[i] = np.sqrt(closest_point_on_triangles(p[None, None, :], t[None, :, 0], t[None, :, 1], t[None, :, 2]).min())
    return out


def approx_unsigned(P, V, F, tree, k=32):
    """Original method: exact distance to the k triangles with the nearest centroids."""
    _, idx = tree.query(P, k=min(k, len(F)))
    idx = np.atleast_2d(idx)
    a, b, c = V[F[idx, 0]], V[F[idx, 1]], V[F[idx, 2]]
    return np.sqrt(closest_point_on_triangles(P[:, None, :], a, b, c).min(axis=1))


# ----------------------------------------------------------------- fitting
def nll(theta, d, y):
    t, logw = theta
    w = np.exp(logw)
    z = (d - t) / w                     # P(inside) = sigmoid(-z)
    # -log L = sum y*log(1+e^z) + (1-y)*log(1+e^-z)
    return float(np.sum(y * np.logaddexp(0, z) + (1 - y) * np.logaddexp(0, -z)))


def grad(theta, d, y):
    t, logw = theta
    w = np.exp(logw)
    z = (d - t) / w
    s = expit(z)                        # = 1 - P(inside)
    r = s - (1 - y)                     # d nll / dz
    return np.array([np.sum(r * (-1 / w)), np.sum(r * (-z))])


def fit_logistic(d, y, band, starts=None):
    y = y.astype(float)
    lo = np.array([-band, np.log(1e-6)]); hi = np.array([band, np.log(band)])
    if starts is None:
        # crude midpoint: where the running inside-fraction crosses 0.5
        order = np.argsort(d); cum = np.cumsum(y[order]) / np.arange(1, len(d) + 1)
        t0 = float(d[order][np.argmin(np.abs(cum - 0.5))]) if len(d) else 0.0
        starts = [(t0, np.log(3e-4)), (0.0, np.log(1e-3)), (band / 4, np.log(1e-4)), (-band / 4, np.log(1e-3)),
                  (0.003, np.log(3e-4))]
    fits = []
    for s in starts:
        x0 = np.clip(np.array(s, float), lo + 1e-9, hi - 1e-9)
        r = minimize(nll, x0, args=(d, y), jac=grad, method="L-BFGS-B", bounds=list(zip(lo, hi)),
                     options={"maxiter": 2000, "ftol": 1e-14, "gtol": 1e-10})
        fits.append(r)
    best = min(fits, key=lambda r: r.fun)
    near = [r for r in fits if r.fun - best.fun < 1e-6 * max(1.0, abs(best.fun))]
    t_spread = float(np.ptp([r.x[0] for r in near])) if len(near) > 1 else 0.0
    t, logw = best.x
    g = grad(best.x, d, y)
    at_lo = best.x <= lo + 1e-3 * (hi - lo); at_hi = best.x >= hi - 1e-3 * (hi - lo)
    pg = np.where((at_lo & (g > 0)) | (at_hi & (g < 0)), 0.0, g)      # projected gradient
    pg = pg * np.array([np.exp(logw), 1.0]) / max(1, len(d))         # scale-free, per point
    n_in, n_out = int(y.sum()), int(len(y) - y.sum())
    # class overlap: any inside point beyond the outermost outside point's inner edge
    overlap = n_in > 0 and n_out > 0 and d[y == 1].max() > d[y == 0].min()
    reasons = []
    if not best.success: reasons.append("optimizer_not_success")
    if np.max(np.abs(pg)) > 1e-4: reasons.append("gradient_not_small")
    if at_lo[0] or at_hi[0]: reasons.append("t50_on_bound")
    if at_lo[1]: reasons.append("width_on_lower_bound")
    if at_hi[1]: reasons.append("width_on_upper_bound")
    if t_spread > 1e-5: reasons.append("starts_disagree")
    if min(n_in, n_out) < 20: reasons.append("too_few_points_in_one_class")
    if not overlap: reasons.append("perfectly_separable")
    # Approximate standard error from expected Fisher information (not the full observed Hessian)
    w = np.exp(logw); z = (d - t) / w; p = expit(z); v = p * (1 - p)
    H = np.array([[np.sum(v) / w**2, np.sum(v * z) / w], [np.sum(v * z) / w, np.sum(v * z * z)]])
    # (logw block uses dz/dlogw = -z; cross terms ignore the residual term, i.e. expected information)
    try:
        se_t = float(np.sqrt(np.linalg.inv(H)[0, 0]))
    except np.linalg.LinAlgError:
        se_t = float("nan")
    return {"t50": float(t), "w": float(w), "nll": float(best.fun), "success": bool(best.success),
            "n_iter": int(best.nit), "max_proj_grad": float(np.max(np.abs(pg))), "t50_spread_across_starts": t_spread,
            "n_starts_at_optimum": len(near), "se_t50": se_t, "n_inside": n_in, "n_outside": n_out,
            "converged": len(reasons) == 0, "fail_reasons": ";".join(reasons)}


# ----------------------------------------------------------------- per object
def process(d, band, max_points, seed, k_approx, brute_force=False):
    mesh, pts, occ, loc, s = load_instance(d)
    Vn = (np.asarray(mesh.vertices, float) - loc) / s          # mesh -> normalized frame
    F = np.asarray(mesh.faces, int)
    cent = Vn[F].mean(axis=1)
    tree = cKDTree(cent)
    # preselect by nearest-vertex distance (an upper bound on the true distance, loosened by edge length)
    edges = np.linalg.norm(Vn[F[:, [1, 2, 0]]] - Vn[F], axis=-1)
    dv, _ = cKDTree(Vn).query(pts)
    pre = np.where(dv < band + edges.max())[0]
    d_apx_all = approx_unsigned(pts[pre], Vn, F, tree, k_approx)
    keep = pre[d_apx_all < band]
    rng = np.random.default_rng(seed)
    if len(keep) > max_points:
        keep = np.sort(rng.choice(keep, max_points, replace=False))
    P, y = pts[keep], occ[keep]
    wn = winding_number(P, Vn, F)
    sign = np.where(wn > 0.5, -1.0, 1.0)                     # inside -> negative
    u_apx = approx_unsigned(P, Vn, F, tree, k_approx)
    d_apx = sign * u_apx
    d_ex = sign * (exact_unsigned(P, Vn, F) if brute_force else exact_unsigned_pruned(P, Vn, F, tree, u_apx))
    diff = np.abs(np.abs(d_apx) - np.abs(d_ex))
    fe = fit_logistic(d_ex, y, band)
    fa = fit_logistic(d_apx, y, band)
    # volume-implied offset
    V_m, A_m = abs(float(mesh.volume)), float(mesh.area)
    praw = pts * s + loc
    box = np.prod(praw.max(axis=0) - praw.min(axis=0))
    V_o = occ.mean() * box
    t_vol = (V_o - V_m) / (A_m * s)
    # label sanity: points clearly inside / outside
    # label sanity relative to the fitted transition: deep inside = d < t50 - 10w, far outside = d > t50 + 10w
    marg = 10 * max(fe["w"], 1e-4)
    deep_in, far_out = d_ex < fe["t50"] - marg, d_ex > fe["t50"] + marg
    row = {"n_faces": len(F), "n_points_used": len(P),
           "frac_deep_inside_labeled_out": float(np.mean(~y[deep_in])) if deep_in.any() else np.nan,
           "frac_far_outside_labeled_in": float(np.mean(y[far_out])) if far_out.any() else np.nan,
           "approx_minus_exact_max": float(diff.max()), "approx_minus_exact_median": float(np.median(diff)),
           "frac_points_approx_differs_gt_1e-6": float(np.mean(diff > 1e-6)),
           "t_vol": float(t_vol), "V_mesh": V_m, "A_mesh": A_m, "V_occ": float(V_o), "scale": s}
    row.update({f"exact_{k}": v for k, v in fe.items()})
    row.update({f"approx_{k}": v for k, v in fa.items()})
    return row


def find_instance_dirs(root):
    return {os.path.basename(os.path.dirname(p)): os.path.dirname(p)
            for p in glob.glob(os.path.join(root, "**", "raw_mesh.off"), recursive=True)
            if "__MACOSX" not in p}


def ids_from_previous(path_or_dir):
    """Return object IDs (and the original per-object results) from the earlier mechanism run."""
    cands = [path_or_dir] if os.path.isfile(path_or_dir) else \
        sorted(glob.glob(os.path.join(path_or_dir, "**", "*.csv"), recursive=True),
               key=lambda q: (0 if re.search(r"mech|boundary|offset", q, re.I) else 1, q))
    for p in cands:
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        idc = next((c for c in df.columns if re.match(r"^(instance_?id|object_?id|id|instance)$", str(c), re.I)), None)
        if idc and 1 <= len(df) <= 500 and any(re.search(r"t50|midpoint|offset", str(c), re.I) for c in df.columns):
            return p, df, idc
    return None


def summarize(df, col_t, col_w):
    t = df[col_t].to_numpy(float); w = df[col_w].to_numpy(float); tv = df["t_vol"].to_numpy(float)
    ok = np.isfinite(t) & np.isfinite(tv)
    q1, q3 = np.percentile(t[ok], [25, 75]) if ok.any() else (np.nan, np.nan)
    return {"n": int(ok.sum()), "t50_median": float(np.median(t[ok])), "t50_iqr": [float(q1), float(q3)],
            "t50_cv": float(np.std(t[ok], ddof=1) / np.mean(t[ok])) if ok.sum() > 1 else np.nan,
            "w_median": float(np.median(w[ok])), "t_vol_median": float(np.median(tv[ok])),
            "ratio_t50_to_tvol": float(np.median(t[ok]) / np.median(tv[ok])),
            "pearson_r": float(pearsonr(t[ok], tv[ok])[0]) if ok.sum() > 2 else np.nan,
            "spearman_rho": float(spearmanr(t[ok], tv[ok])[0]) if ok.sum() > 2 else np.nan,
            "t50_over_half_ray": float(np.median(t[ok]) / PIPELINE["half_ray_offset"]),
            "t50_over_full_ray": float(np.median(t[ok]) / PIPELINE["full_ray_offset"])}


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--shard-root", required=True, help="extracted 10_24553_27272 folder")
    ap.add_argument("--previous", help="earlier mechanism_offset output (CSV or folder) to reuse its 60 IDs")
    ap.add_argument("--n", type=int, default=60, help="objects to sample if --previous is not given")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--band", type=float, default=0.02, help="max |distance| of selected points (normalized)")
    ap.add_argument("--max-points", type=int, default=3000)
    ap.add_argument("--k-approx", type=int, default=32)
    ap.add_argument("--brute-force", action="store_true", help="exact distance over all faces (slow; for checking)")
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    dirs = find_instance_dirs(a.shard_root)
    if not dirs:
        sys.exit(f"No raw_mesh.off files found under {a.shard_root}")
    prev = ids_from_previous(a.previous) if a.previous else None
    if prev:
        ppath, pdf_, idc = prev
        ids = [str(x) for x in pdf_[idc]]
        print(f"Reusing {len(ids)} IDs from {ppath}")
    else:
        if a.previous:
            raise ValueError("No usable prior object table; refusing to silently change the boundary sample")
        ids = sorted(np.random.default_rng(a.seed).choice(sorted(dirs), size=min(a.n, len(dirs)), replace=False))
        pdf_ = None
    missing = [i for i in ids if i not in dirs]
    if missing:
        print(f"WARNING: {len(missing)} IDs not found in the shard, e.g. {missing[:5]}")
    ids = [i for i in ids if i in dirs]

    out_csv = os.path.join(a.out, "e5_refit_per_object.csv")
    done = pd.read_csv(out_csv, dtype={"id": str}) if os.path.exists(out_csv) else pd.DataFrame()
    rows = done.to_dict("records")
    have = set(done["id"]) if len(done) else set()
    for j, i in enumerate(ids):
        if i in have:
            continue
        t0 = time.time()
        try:
            r = process(dirs[i], a.band, a.max_points, a.seed + j, a.k_approx, a.brute_force)
            r["id"] = i; r["error"] = ""
        except Exception as e:                      # keep going; record the failure
            r = {"id": i, "error": repr(e), "exact_converged": False, "exact_fail_reasons": "processing_error"}
        rows.append(r)
        pd.DataFrame(rows).to_csv(out_csv, index=False)   # checkpoint after every object
        print(f"[{j + 1}/{len(ids)}] {i}: t50={r.get('exact_t50', float('nan')):.5f} "
              f"converged={r.get('exact_converged')} {r.get('exact_fail_reasons', '')} ({time.time() - t0:.1f}s)")

    df = pd.DataFrame(rows)
    df["id"] = df["id"].astype(str)
    conv = df[df["exact_converged"] == True]
    fails = df[df["exact_converged"] != True]
    reasons = {}
    for s in fails.get("exact_fail_reasons", pd.Series(dtype=str)).fillna(""):
        for k in filter(None, str(s).split(";")):
            reasons[k] = reasons.get(k, 0) + 1
    summary = {
        "run_utc": datetime.now(timezone.utc).isoformat(), "settings": vars(a),
        "n_objects": len(df), "n_converged": len(conv), "n_failed": len(fails), "fail_reason_counts": reasons,
        "failed_ids": fails["id"].tolist(),
        "exact_all": summarize(df.dropna(subset=["exact_t50"]), "exact_t50", "exact_w") if "exact_t50" in df else None,
        "exact_converged_only": summarize(conv, "exact_t50", "exact_w") if len(conv) > 2 else None,
        "approx_converged_only": summarize(df[df.get("approx_converged") == True], "approx_t50", "approx_w")
        if "approx_converged" in df and (df["approx_converged"] == True).sum() > 2 else None,
        "approx_vs_exact_distance": {
            "max_abs_diff": float(df["approx_minus_exact_max"].max()),
            "median_of_object_medians": float(df["approx_minus_exact_median"].median()),
            "median_frac_points_differing": float(df["frac_points_approx_differs_gt_1e-6"].median()),
            "t50_abs_change_median": float((df["approx_t50"] - df["exact_t50"]).abs().median()),
            "t50_abs_change_max": float((df["approx_t50"] - df["exact_t50"]).abs().max())},
        "label_sanity": {"max_frac_deep_inside_labeled_out": float(df["frac_deep_inside_labeled_out"].max()),
                         "max_frac_far_outside_labeled_in": float(df["frac_far_outside_labeled_in"].max())},
        "original_reported": ORIGINAL, "pipeline_reference": PIPELINE,
    }
    if pdf_ is not None:
        tcol = next((c for c in pdf_.columns if re.search(r"t50|midpoint", str(c), re.I)), None)
        if tcol:
            m = df.merge(pdf_[[idc, tcol]].rename(columns={idc: "id", tcol: "t50_original"}).astype({"id": str}), on="id")
            summary["vs_original_run"] = {"n_matched": len(m),
                                          "median_abs_change_t50": float((m["exact_t50"] - m["t50_original"]).abs().median()),
                                          "max_abs_change_t50": float((m["exact_t50"] - m["t50_original"]).abs().max())}
    json.dump(summary, open(os.path.join(a.out, "e5_refit_summary.json"), "w"), indent=2, default=float)

    s = summary["exact_converged_only"] or summary["exact_all"]
    ad = summary["approx_vs_exact_distance"]
    print("\n=== E5 refit ===")
    print(f"objects {len(df)}, converged {len(conv)}, failed {len(fails)} {reasons if reasons else ''}")
    print(f"converged fits: t50 median {s['t50_median']:.5f} (IQR {s['t50_iqr'][0]:.5f}-{s['t50_iqr'][1]:.5f}, "
          f"CV {s['t50_cv']:.2f}), w median {s['w_median']:.5f}")
    print(f"volume-implied offset median {s['t_vol_median']:.5f}; ratio {s['ratio_t50_to_tvol']:.2f}; "
          f"per-object r {s['pearson_r']:.3f}")
    print(f"t50 / half-ray reference {s['t50_over_half_ray']:.2f}; / full-ray {s['t50_over_full_ray']:.2f}")
    print(f"approx vs exact distance: max diff {ad['max_abs_diff']:.2e}; median t50 change {ad['t50_abs_change_median']:.2e}")
    print(f"original paper values: {ORIGINAL}")
    print("\nSuggested Methods/Limitations wording:")
    print(f"  \"Each fit was accepted only if the optimizer converged, the projected gradient was small, neither "
          f"parameter lay on a bound, and five starting points agreed on t50; {len(conv)} of {len(df)} fits met all "
          f"criteria. Distances were computed exactly (all faces, winding-number sign); the 32-centroid approximation "
          f"differed from the exact distance by at most {ad['max_abs_diff']:.1e} normalized units and changed t50 by a "
          f"median of {ad['t50_abs_change_median']:.1e}.\"")


if __name__ == "__main__":
    main()


## 1–2. Figure re-export and coverage check

The script finds the per-object table and the saved bounds by itself and prints which columns and files it used. **Check that printout.** If a column is wrong, list the columns with the next cell and pass them explicitly, e.g. `--col-uncorrected err_raw_pct --col-global err_global_pct --col-occ-only err_occ_pct`.

In [ ]:
# Optional: list candidate per-object tables and their columns
import pandas as pd
for p in sorted(glob.glob(os.path.join(FROZEN_DIR, '**', '*.csv'), recursive=True)):
    try:
        df = pd.read_csv(p, nrows=3); n = sum(1 for _ in open(p)) - 1
        if n >= 2000: print(f'{p}  (n={n})\n   {list(df.columns)}\n')
    except Exception as e:
        pass

In [ ]:
!python /content/frozen_figure_and_coverage.py \
    --frozen-dir "$FROZEN_DIR" \
    --dev-dir "$OUT_ROOT" \
    --out "$OUT/frozen_figure_and_coverage"
# add --col-... / --bound-... / --global-offset overrides here if the printout shows a wrong pick

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(OUT, 'frozen_figure_and_coverage', 'frozen_errors.png'), width=420))

## 3. E5 refit with convergence check

In [ ]:
# Extract the development shard to local disk if it is not already there.
# The development shard is recognised by its first instance folder, 24553.
def find_dev_shard():
    hits = [p for p in glob.glob(f'{DATA_DIR}/**/24553/raw_mesh.off', recursive=True) if '__MACOSX' not in p]
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

SHARD = find_dev_shard()
if SHARD is None:
    os.makedirs(DATA_DIR, exist_ok=True)
    !unzip -q -n "$DEV_ZIP" -d "$DATA_DIR"
    SHARD = find_dev_shard()
print('shard root:', SHARD, '| instances:', len(glob.glob(f'{SHARD}/*/raw_mesh.off')))

In [ ]:
# Resume-safe: rerun this cell after a disconnect with the same OUT (set OUT to the earlier folder first).
!python /content/e5_boundary_refit.py \
    --shard-root "$SHARD" \
    --previous "$PREV_DIR" \
    --out "$OUT/e5_refit"
# Missing prior IDs now stop the run; the original sample must be preserved.


In [ ]:
import json
s = json.load(open(os.path.join(OUT, 'e5_refit', 'e5_refit_summary.json')))
print('converged:', s['n_converged'], '/', s['n_objects'], '| failures:', s['fail_reason_counts'])
print(json.dumps({k: s[k] for k in ('exact_converged_only', 'approx_vs_exact_distance', 'label_sanity')}, indent=1))
print(json.dumps(s.get('vs_original_run', {}), indent=1))

## What to bring back
- `frozen_figure_and_coverage/frozen_errors.pdf` → Overleaf `figures/`. Change `frozen_errors.png` to `frozen_errors.pdf` in `main.tex`.
- The coverage printout (or `coverage_check.json`). If it says **IDENTICAL covered sets**, add to the Table IV discussion: *"Because nearly all raw errors are positive, the global correction and the uncorrected estimate cover the same objects, so their coverage values coincide."*
- `e5_refit/e5_refit_summary.json`. Its printed "Suggested wording" replaces the E5 limitation sentence. If any fits failed, report the count and whether the medians change when they are excluded.